# Bài 2 (bản phân lớp) — Feature Selection dựa trên Correlation cho bài toán phân lớp tuyến tính

**Bài toán phân lớp:** giống Bài 1 — dự đoán một địa chỉ Bitcoin là `white` hay thuộc họ ransomware nào
(**7 lớp**: `white`, 5 họ lớn nhất, `otherRansom`) trên bộ dữ liệu *BitcoinHeist* (UCI, 2.916.697 dòng).

**Mô hình:** đề yêu cầu **Linear Regression** và độ đo **MAE**. Ta dùng Linear Regression như một **bộ phân lớp
tuyến tính** (*least-squares classification*): mã hoá nhãn thành vector one-hot, hồi quy tuyến tính từng thành phần,
lớp dự đoán là thành phần có điểm lớn nhất. Đây chính là `RidgeClassifier` của scikit-learn (khi hệ số L2 → 0).

| | Bản hồi quy (`bai2.ipynb`) | **Bản phân lớp (notebook này)** |
|---|---|---|
| Mục tiêu | `log_income` (số liên tục) | **nhãn 7 lớp của Bài 1** |
| Linear Regression | dự đoán một số | dự đoán 7 điểm (one-hot) → `argmax` → lớp |
| Tương quan dùng để chọn | Pearson giữa đặc trưng và `log_income` | **Point-biserial** giữa đặc trưng và từng lớp (+ tỉ số tương quan η) |
| Độ đo | MAE | **MAE** (trên điểm one-hot) + accuracy, macro F1 |

**Nội dung**
1. Đọc dữ liệu & tạo đặc trưng
2. Linear Regression như một bộ phân lớp tuyến tính
3. Dữ liệu thử nghiệm: vì sao cần cân bằng lớp
4. **(a)** Chọn đặc trưng dựa trên tương quan — các đồ thị minh hoạ
5. **(b)** Thử nghiệm các tập đặc trưng với Linear Regression (phân lớp) — so sánh MAE
6. Kết luận

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeClassifier
from sklearn.feature_selection import r_regression, f_classif, SelectKBest
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

DATA_PATH = "BitcoinHeistData.csv"
OUT_DIR = "outputs/bai2_cls"
RANDOM_STATE = 42
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Đọc dữ liệu & tạo đặc trưng

Nhãn và đặc trưng **giống hệt Bài 1**:

| Nhóm | Đặc trưng | Xử lý |
|---|---|---|
| Gốc | `length, weight, count, looped, neighbors, income` | `log1p` (lệch mạnh) |
| Tỉ lệ (tạo mới) | `income_per_count, income_per_neighbor, weight_per_count, looped_ratio` | `log1p` |
| Độ "tròn" của số tiền | `income_tz` = số chữ số 0 ở cuối của `income` | giữ nguyên |
| Thời gian | `year, day, month, weekday` | giữ nguyên (số) |

→ **15 đặc trưng số**. Khác bản hồi quy, ở đây mục tiêu là **nhãn** nên các đặc trưng tạo từ `income` **không** gây
rò rỉ dữ liệu và được giữ lại. `address` bị loại vì là định danh.

In [ ]:
t0 = time.perf_counter()
df = pd.read_csv(DATA_PATH)
date = pd.to_datetime(df["year"].astype(str) + "-01-01") + pd.to_timedelta(df["day"] - 1, unit="D")
df["month"] = date.dt.month
df["weekday"] = date.dt.dayofweek
inc_int = df["income"].round().astype("int64")
df["income_tz"] = sum((inc_int % 10**k == 0).astype("int8") for k in range(1, 13))
df["income_per_count"] = df["income"] / df["count"]
df["income_per_neighbor"] = df["income"] / df["neighbors"].clip(lower=1)
df["weight_per_count"] = df["weight"] / df["count"]
df["looped_ratio"] = df["looped"] / df["count"]

WHITE = "white"
TOP_FAMILIES = df.loc[df["label"] != WHITE, "label"].value_counts().head(5).index.tolist()
df["target"] = np.where(df["label"] == WHITE, WHITE,
                np.where(df["label"].isin(TOP_FAMILIES), df["label"], "otherRansom"))
CLASSES = sorted(df["target"].unique())          # thứ tự lớp cố định (như Bài 1)
K = len(CLASSES)
W = CLASSES.index(WHITE)

SKEWED = ["length", "weight", "count", "looped", "neighbors", "income",
          "income_per_count", "income_per_neighbor", "weight_per_count", "looped_ratio"]
OTHER = ["income_tz", "year", "day", "month", "weekday"]
FEATURES = SKEWED + OTHER
X = pd.concat([np.log1p(df[SKEWED]), df[OTHER]], axis=1)[FEATURES].astype(float)
y = df["target"].map({c: i for i, c in enumerate(CLASSES)}).rename("target")
print(f"Đọc & tạo đặc trưng trong {time.perf_counter() - t0:.1f}s — {len(X):,} dòng × {X.shape[1]} đặc trưng, {K} lớp")
df["target"].value_counts().to_frame("số dòng").assign(pct=lambda d: 100 * d["số dòng"] / len(df))

## 2. Linear Regression như một bộ phân lớp tuyến tính

**Mã hoá one-hot:** nhãn $y = k$ được biểu diễn bằng vector $\mathbf{Y} \in \{0,1\}^K$ với $Y_k = 1$, các thành phần
khác bằng 0.

**Huấn luyện:** `LinearRegression` hồi quy đồng thời $K$ đầu ra (bình phương tối thiểu):

$$\hat s_k(\mathbf{x}) = w_{k0} + \mathbf{w}_k^\top \mathbf{x}, \qquad k = 1,\dots,K$$

Vì $\mathbb{E}[Y_k \mid \mathbf{x}] = P(y = k \mid \mathbf{x})$, điểm $\hat s_k$ là một **xấp xỉ tuyến tính của xác
suất** thuộc lớp $k$.

**Dự đoán:** $\hat y = \arg\max_k \hat s_k(\mathbf{x})$. Ranh giới giữa hai lớp $j, k$ là tập
$\hat s_j(\mathbf{x}) = \hat s_k(\mathbf{x})$ — một **siêu phẳng** → đây là bộ phân lớp **tuyến tính**.

**Độ đo MAE** (càng nhỏ càng tốt) đo khoảng cách giữa điểm dự đoán và nhãn one-hot:

$$\text{MAE} = \frac{1}{nK}\sum_{i=1}^{n}\sum_{k=1}^{K}\big|Y_{ik} - \hat s_k(\mathbf{x}_i)\big|$$

(`mean_absolute_error(Y, S)` với `Y`, `S` là ma trận $n \times K$). *Lưu ý:* nếu tính MAE trên nhãn cứng (one-hot của
$\hat y$) thì $\text{MAE} = \tfrac{2}{K}(1 - \text{accuracy})$, tức chỉ là tỉ lệ lỗi nhân hằng số; MAE trên **điểm**
nhạy hơn vì phản ánh cả độ "tự tin" của mô hình. Ta báo cáo thêm **accuracy** và **macro F1** (như Bài 1).

**Mốc so sánh (baseline):** không dùng đặc trưng nào — điểm luôn bằng tỉ lệ các lớp trên tập train
(và luôn dự đoán lớp nhiều nhất).

In [ ]:
def onehot(labels):
    return np.eye(K)[np.asarray(labels)]

def make_lr():
    # Chuẩn hoá không đổi dự đoán của Linear Regression nhưng giúp so sánh độ lớn các hệ số
    return Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())])

## 3. Dữ liệu thử nghiệm: vì sao cần cân bằng lớp

Thử trực tiếp trên **phân phối thật** (98,6% `white`): chia 80/20 như Bài 1, train Linear Regression (one-hot) với
cả 15 đặc trưng.

In [ ]:
X_tr_full, X_te_full, y_tr_full, y_te_full = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
pred_full = make_lr().fit(X_tr_full, onehot(y_tr_full)).predict(X_te_full).argmax(1)
print(f"Dự đoán 'white' cho {100 * (pred_full == W).mean():.2f}% số dòng test | "
      f"accuracy = {accuracy_score(y_te_full, pred_full):.4f} | macro F1 = {f1_score(y_te_full, pred_full, average='macro'):.4f}")
pd.Series(pred_full).map(dict(enumerate(CLASSES))).value_counts().to_frame("số dòng dự đoán")

Trên phân phối thật, bộ phân lớp bình phương tối thiểu gần như **luôn dự đoán `white`** (accuracy ≈ 98,6% chỉ bằng
tỉ lệ `white`, macro F1 rất thấp): lớp `white` chi phối hàm mất mát, điểm của các lớp ransomware không bao giờ vượt
điểm `white`. Khi đó **mọi tập đặc trưng đều cho kết quả gần như nhau** → không thể so sánh các tập đặc trưng.

**Giải pháp (như hướng undersampling của Bài 1):** tạo tập thử nghiệm **cân bằng** gồm **toàn bộ** dòng ransomware
và **cùng số lượng** dòng `white` lấy ngẫu nhiên → `white` chiếm 50%, 6 lớp ransomware chiếm 50% còn lại. Chia 80%
train / 20% test (stratified). **Tương quan và việc chọn đặc trưng chỉ tính trên tập train**; tập test chỉ dùng để đo
MAE.

In [ ]:
del X_tr_full, X_te_full, y_tr_full, y_te_full, pred_full
rs = np.random.RandomState(RANDOM_STATE)
ransom_idx = y.index[y != W]
white_idx = rs.choice(y.index[y == W], len(ransom_idx), replace=False)
idx = np.concatenate([white_idx, ransom_idx])
X_train, X_test, y_train, y_test = train_test_split(X.loc[idx], y.loc[idx], test_size=0.2, stratify=y.loc[idx],
                                                    random_state=RANDOM_STATE)
Y_train, Y_test = onehot(y_train), onehot(y_test)
IND = pd.DataFrame(Y_train, index=X_train.index, columns=CLASSES)   # biến chỉ thị 1[y = k] của từng lớp (tập train)

sizes = pd.concat([s.map(dict(enumerate(CLASSES))).value_counts().rename(n)
                   for n, s in [("train", y_train), ("test", y_test)]], axis=1)
sizes["% (train)"] = 100 * sizes["train"] / sizes["train"].sum()
sizes.loc["TỔNG"] = sizes.sum()
sizes

## 4. (a) Chọn đặc trưng dựa trên tương quan (Correlation-based Feature Selection)

Với bài toán phân lớp, mục tiêu là **nhãn danh nghĩa** (không có thứ tự) → không tính Pearson trực tiếp với "số thứ
tự lớp" được. Ta tính tương quan giữa đặc trưng $X_j$ và **biến chỉ thị** của từng lớp $Y_k = \mathbb{1}[y = k]$ —
gọi là **tương quan point-biserial** (chính là Pearson khi một biến là nhị phân):

$$r_{jk} = \operatorname{corr}(X_j, Y_k) = \frac{\bar x_{j,\,y=k} - \bar x_{j,\,y\neq k}}{\sigma_{X_j}}\sqrt{p_k(1-p_k)}$$

($p_k$: tỉ lệ lớp $k$). $|r_{jk}|$ lớn ⇔ trung bình của $X_j$ trong lớp $k$ khác hẳn các lớp còn lại ⇔ $X_j$ giúp
**tách lớp $k$ khỏi các lớp khác** — đúng việc mà đầu ra thứ $k$ của Linear Regression phải làm (hồi quy 1 biến:
$R^2 = r_{jk}^2$).

**Điểm tương quan tổng hợp** của đặc trưng $j$ trên cả $K$ lớp (căn trung bình bình phương):

$$R_j = \sqrt{\frac{1}{K}\sum_{k=1}^{K} r_{jk}^2}$$

**Hai nguyên tắc chọn đặc trưng** (như bản hồi quy):
1. **Liên quan (relevance):** giữ các đặc trưng có $R_j$ lớn — **Top-k** hoặc **ngưỡng** $R_j \ge t$.
2. **Không dư thừa (redundancy):** nếu hai đặc trưng tương quan rất mạnh *với nhau* ($|r(X_i, X_j)| \ge$ ngưỡng),
   bỏ đặc trưng có $R$ thấp hơn.

### 4.1 Tương quan giữa từng đặc trưng và từng lớp

In [ ]:
corr_cls = pd.DataFrame({c: X_train.corrwith(IND[c]) for c in CLASSES})        # r_jk: đặc trưng × lớp
score = pd.DataFrame({"R": np.sqrt((corr_cls ** 2).mean(axis=1)),
                      "max |r|": corr_cls.abs().max(axis=1),
                      "lớp có |r| lớn nhất": corr_cls.abs().idxmax(axis=1)}).sort_values("R", ascending=False)
score.insert(0, "hạng", range(1, len(score) + 1))
RANKING = score.index.tolist()          # đặc trưng xếp theo R giảm dần
score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5), gridspec_kw={"width_ratios": [1.6, 1]})
sns.heatmap(corr_cls.loc[RANKING], annot=True, fmt="+.2f", cmap="RdBu_r", vmin=-0.45, vmax=0.45, center=0,
            cbar_kws={"label": "r (point-biserial)"}, annot_kws={"size": 8}, ax=axes[0])
axes[0].set(title="Tương quan point-biserial r(đặc trưng, 1[y = lớp]) — tập train\n(xếp theo R giảm dần)", xlabel="", ylabel="")
axes[0].tick_params(axis="x", rotation=30)
R = score["R"][::-1]
axes[1].barh(R.index, R.values, color="tab:blue")
for i, v in enumerate(R.values):
    axes[1].text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
axes[1].set(title="Điểm tương quan tổng hợp R = √(trung bình r² theo lớp)", xlabel="R", xlim=(0, R.max() * 1.18))
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/corr_with_classes.png", dpi=120); plt.show()

In [ ]:
smp = pd.concat([X_train, y_train.map(dict(enumerate(CLASSES))).rename("lớp")], axis=1)
order = [WHITE] + [c for c in CLASSES if c != WHITE]
picks = [RANKING[0], RANKING[1], RANKING[-1]]
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for ax, feat, color in zip(axes, picks, ["tab:blue", "tab:green", "tab:gray"]):
    sns.boxplot(data=smp, x="lớp", y=feat, order=order, ax=ax, color=color, fliersize=0, showmeans=True,
                meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black"})
    ax.set(title=f"{feat}: R = {score.loc[feat, 'R']:.3f} (hạng {score.loc[feat, 'hạng']})", xlabel="",
           ylabel=feat + (" (log1p)" if feat in SKEWED else ""))
    ax.tick_params(axis="x", rotation=35)
plt.suptitle("Minh hoạ: phân phối đặc trưng theo lớp — R cao: trung bình (chấm trắng) khác nhau rõ giữa các lớp; "
             "R thấp: gần như giống nhau")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/corr_illustration.png", dpi=120); plt.show()

**Nhận xét (4.1):**
- `income_tz` (độ "tròn" của số tiền) có $R$ lớn nhất: tương quan **âm** mạnh với `white` và dương với nhiều họ
  ransomware → địa chỉ ransomware hay nhận các khoản tiền "tròn" (tiền chuộc được định giá sẵn).
- `year` đứng thứ hai: mỗi họ ransomware hoạt động trong một giai đoạn riêng (`montrealCryptoLocker` 2013,
  `paduaCryptoWall` 2014–2015, `princetonLocky`/`princetonCerber` 2016–2017), nên `year` tách tốt các **họ**
  ransomware với nhau (r = −0,31 với CryptoLocker, +0,31 với Cerber) dù gần như không tách `white`.
- Nhóm `income`, `income_per_neighbor`, `neighbors`, `weight`, `month`/`day` ở mức trung bình; `count`, `length`,
  `weight_per_count`, `weekday` gần như không tách được lớp nào ($R < 0{,}05$).
- Mọi $|r|$ đều nhỏ (< 0,45) → từng đặc trưng riêng lẻ chỉ tách lớp ở mức **yếu**.

### 4.2 Tương quan giữa các đặc trưng — phát hiện đặc trưng dư thừa

Thuật toán (tham lam): duyệt các đặc trưng theo $R$ **giảm dần**; giữ một đặc trưng nếu nó **không** tương quan mạnh
($|r| \ge$ ngưỡng) với bất kỳ đặc trưng nào đã giữ trước đó.

In [ ]:
corr_ff = X_train.corr()
pairs = (corr_ff.where(np.triu(np.ones(corr_ff.shape, dtype=bool), k=1)).stack()
         .rename("r").reset_index().rename(columns={"level_0": "đặc trưng 1", "level_1": "đặc trưng 2"}))
pairs = pairs.reindex(pairs["r"].abs().sort_values(ascending=False).index).reset_index(drop=True)

def drop_redundant(features, thr):
    kept = []
    for f in sorted(features, key=RANKING.index):          # đặc trưng liên quan nhất được xét trước
        if all(abs(corr_ff.loc[f, g]) < thr for g in kept):
            kept.append(f)
    return kept

for thr in (0.95, 0.9, 0.8):
    print(f"Ngưỡng dư thừa {thr}: bỏ {[f for f in RANKING if f not in drop_redundant(FEATURES, thr)]}")
pairs.head(8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={"width_ratios": [1.25, 1]})
c = corr_ff.loc[RANKING, RANKING]
sns.heatmap(c, mask=np.triu(np.ones_like(c, dtype=bool), k=1), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.7}, annot_kws={"size": 7}, ax=axes[0])
axes[0].set_title("Ma trận tương quan Pearson giữa các đặc trưng (tập train, xếp theo R)")
a, b = pairs.loc[1, "đặc trưng 1"], pairs.loc[1, "đặc trưng 2"]          # cặp mạnh thứ hai (cặp đầu là day – month)
hb = axes[1].hexbin(X_train[a], X_train[b], gridsize=50, bins="log", cmap="viridis", mincnt=1)
axes[1].set(xlabel=a + " (log1p)", ylabel=b + " (log1p)", title=f"{a} – {b}: r = {corr_ff.loc[a, b]:+.3f}")
fig.colorbar(hb, ax=axes[1], label="số dòng (log)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/redundancy.png", dpi=120); plt.show()

**Nhận xét (4.2):**
- `day` – `month` (r ≈ 0,996): `month` suy ra trực tiếp từ `day` → một trong hai là dư thừa.
- `income` – `income_per_neighbor` (r ≈ 0,95): phần lớn địa chỉ chỉ có 1 giao dịch đến, khi đó hai đặc trưng bằng
  nhau.
- `count` – `income_per_count`, `looped` – `looped_ratio`, `length` – `count` cũng tương quan mạnh.
- Câu (b) sẽ kiểm tra: bỏ các đặc trưng này thì độ chính xác có giảm không.

### 4.3 Đối chiếu với `sklearn.feature_selection`

- `r_regression(X, Y_k)`: hệ số Pearson của từng đặc trưng với biến chỉ thị lớp $k$ — phải trùng $r_{jk}$ ở trên.
- `f_classif`: kiểm định **ANOVA F** cho phân lớp (bản tương ứng của `f_regression`). Nó gắn với **tỉ số tương quan**
  $\eta$ (*correlation ratio*) — mở rộng của hệ số tương quan cho biến mục tiêu danh nghĩa:

$$\eta_j^2 = \frac{\sum_k n_k(\bar x_{j,k} - \bar x_j)^2}{\sum_i (x_{ij} - \bar x_j)^2}, \qquad
F_j = \frac{\eta_j^2/(K-1)}{(1-\eta_j^2)/(n-K)}$$

  $F$ tăng theo $\eta$ → `SelectKBest(f_classif, k)` chọn $k$ đặc trưng có $\eta$ lớn nhất.

In [ ]:
assert np.allclose(np.column_stack([r_regression(X_train, IND[c]) for c in CLASSES]), corr_cls.values)
print("r_regression trùng với r_jk tính thủ công ✓")

F, pval = f_classif(X_train, y_train)
n_tr = len(y_train)
eta = pd.Series(np.sqrt(F * (K - 1) / (F * (K - 1) + n_tr - K)), index=FEATURES)
ETA_RANKING = eta.sort_values(ascending=False).index.tolist()
cmp_sk = pd.DataFrame({"R (point-biserial)": score["R"], "hạng theo R": score["hạng"],
                       "η (từ f_classif)": eta, "hạng theo η": eta.rank(ascending=False).astype(int),
                       "F": pd.Series(F, index=FEATURES), "p-value": pd.Series(pval, index=FEATURES)}).loc[RANKING]

overlap = [len(set(RANKING[:k]) & set(X_train.columns[SelectKBest(f_classif, k=k).fit(X_train, y_train).get_support()]))
           for k in range(1, len(FEATURES) + 1)]
print("Số đặc trưng chung giữa Top-k theo R và SelectKBest(f_classif, k):",
      {k: f"{o}/{k}" for k, o in zip(range(1, len(FEATURES) + 1), overlap)})
cmp_sk.style.format({"R (point-biserial)": "{:.4f}", "η (từ f_classif)": "{:.4f}", "F": "{:,.0f}", "p-value": "{:.2e}"})

In [ ]:
ax = cmp_sk[["R (point-biserial)", "η (từ f_classif)"]].plot.bar(figsize=(13, 4.5), color=["tab:blue", "tab:orange"])
ax.set(title="Hai thước đo tương quan đặc trưng – lớp: R (point-biserial, tuyến tính theo từng lớp) và η (ANOVA)",
       ylabel="giá trị", xlabel="")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/R_vs_eta.png", dpi=120); plt.show()

**Nhận xét (4.3):** `r_regression` cho đúng các hệ số point-biserial. $\eta$ luôn $\ge R$ (η đo mọi khác biệt giữa
trung bình các lớp, R chỉ đo theo kiểu "một lớp so với phần còn lại"), nhưng **thứ hạng gần như giống nhau** → Top-k
theo $R$ và `SelectKBest(f_classif, k)` chọn gần như cùng tập (khác biệt chủ yếu ở thứ tự `year` / `income_tz` và
`income` / `neighbors`). Tất cả p-value đều rất nhỏ (< 0,05) → p-value **không** giúp phân biệt; cần dựa vào **độ
lớn** tương quan và kiểm chứng bằng MAE ở câu (b).

## 5. (b) Thử nghiệm các tập đặc trưng với Linear Regression (phân lớp) — so sánh MAE

- **Mô hình:** `StandardScaler` → `LinearRegression` trên nhãn one-hot (7 đầu ra) → `argmax` (mục 2).
- **Độ đo chính:** MAE giữa điểm dự đoán và nhãn one-hot trên tập test (càng nhỏ càng tốt). Kèm **accuracy**,
  **macro F1** và thời gian training.
- **% cải thiện** = mức giảm MAE so với baseline, lấy mức giảm khi dùng **toàn bộ 15 đặc trưng** làm 100%.

### 5.0 Baseline, toàn bộ đặc trưng, và kiểm tra tương đương với `RidgeClassifier`

In [ ]:
def evaluate(name, cols):
    cols = list(cols)
    if cols:
        model = make_lr()
        t0 = time.perf_counter(); model.fit(X_train[cols], Y_train); t_train = time.perf_counter() - t0
        S = model.predict(X_test[cols])
    else:                                       # baseline: điểm = tỉ lệ các lớp trên tập train
        model, t_train, S = None, 0.0, np.tile(Y_train.mean(axis=0), (len(y_test), 1))
    pred = S.argmax(axis=1)
    return {"tập đặc trưng": name, "số đặc trưng": len(cols), "MAE": mean_absolute_error(Y_test, S),
            "accuracy": accuracy_score(y_test, pred), "macro F1": f1_score(y_test, pred, average="macro"),
            "train_time_s": t_train, "đặc trưng": ", ".join(cols)}, model

def improvement(mae):
    return 100 * (BASE["MAE"] - mae) / (BASE["MAE"] - ALL["MAE"])

BASE, _ = evaluate("Baseline (không dùng đặc trưng)", [])
ALL, model_all = evaluate(f"Tất cả {len(FEATURES)} đặc trưng", FEATURES)

rc = Pipeline([("scaler", StandardScaler()), ("rc", RidgeClassifier(alpha=1e-8))]).fit(X_train, y_train)
agree = (rc.predict(X_test) == model_all.predict(X_test).argmax(axis=1)).mean()
print(f"LinearRegression (one-hot + argmax) và RidgeClassifier(alpha≈0) dự đoán trùng nhau trên {100 * agree:.2f}% tập test")
pd.DataFrame([BASE, ALL]).set_index("tập đặc trưng")

### 5.1 Top-k: thêm dần đặc trưng theo thứ tự $R$ giảm dần

In [ ]:
topk = pd.DataFrame([evaluate(f"Top-{k}", RANKING[:k])[0] for k in range(len(RANKING) + 1)])
topk.loc[0, "tập đặc trưng"] = "Top-0 (baseline)"
topk.insert(1, "đặc trưng thêm vào", ["—"] + RANKING)
topk["% cải thiện"] = improvement(topk["MAE"])
GOAL = 90                                   # k nhỏ nhất đạt ≥ 90% mức cải thiện MAE
K_BEST = int(topk.loc[topk["% cải thiện"] >= GOAL, "số đặc trưng"].min())
print(f"K_BEST = {K_BEST}: {RANKING[:K_BEST]}")
print(f"(nếu đòi ≥ 95%: k = {int(topk.loc[topk['% cải thiện'] >= 95, 'số đặc trưng'].min())})")
N_CTRL = len(FEATURES) - K_BEST             # số đặc trưng bị Top-K_BEST loại bỏ (dùng cho đối chứng ở 5.4)
topk.drop(columns="đặc trưng").set_index("tập đặc trưng")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(19, 5.5))
ax = axes[0]
ax.plot(topk["số đặc trưng"], topk["MAE"], marker="o", lw=2)
for _, row in topk.iloc[1:].iterrows():
    ax.annotate(row["đặc trưng thêm vào"], (row["số đặc trưng"], row["MAE"]), textcoords="offset points",
                xytext=(4, 8), fontsize=8, rotation=35)
best = topk.loc[K_BEST]
ax.scatter([K_BEST], [best["MAE"]], s=220, facecolors="none", edgecolors="tab:red", lw=2, zorder=5)
ax.annotate(f"k = {K_BEST}: MAE = {best['MAE']:.4f}\nđạt {best['% cải thiện']:.0f}% mức cải thiện\n"
            f"với {100 * K_BEST / len(FEATURES):.0f}% số đặc trưng", (K_BEST, best["MAE"]),
            textcoords="offset points", xytext=(-60, 110), color="tab:red", arrowprops={"arrowstyle": "->", "color": "tab:red"})
ax.axhline(ALL["MAE"], ls="--", c="tab:green", lw=1, label=f"Tất cả {len(FEATURES)} đặc trưng: MAE = {ALL['MAE']:.4f}")
ax.axhline(BASE["MAE"], ls="--", c="gray", lw=1, label=f"Baseline: MAE = {BASE['MAE']:.4f}")
ax.set(title="MAE trên tập test theo số đặc trưng (Top-k theo R)", xlabel="k — số đặc trưng", ylabel="MAE",
       xticks=range(len(RANKING) + 1))
ax.legend(loc="upper right")
ax = axes[1]
ax.plot(topk["số đặc trưng"], topk["accuracy"], marker="o", lw=2, label="accuracy")
ax.plot(topk["số đặc trưng"], topk["macro F1"], marker="s", lw=2, label="macro F1", color="tab:orange")
ax.axvline(K_BEST, ls=":", c="tab:red", lw=1.5, label=f"k = {K_BEST}")
ax.set(title="Accuracy và macro F1 trên tập test theo số đặc trưng", xlabel="k — số đặc trưng",
       xticks=range(len(RANKING) + 1), ylim=(0, 0.7))
ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/mae_topk.png", dpi=120); plt.show()

**Nhận xét (5.1):**
- Chỉ `income_tz` đã đạt **71%** mức cải thiện MAE: nó tách `white` khỏi ransomware nên điểm $\hat s_{\text{white}}$
  sát nhãn hơn hẳn. Nhưng macro F1 vẫn bằng baseline (0,095) vì một đặc trưng chưa đủ để **phân biệt các họ**
  ransomware; thêm `year` làm MAE nhích lên nhẹ (0,1844 → 0,1849) nhưng macro F1 tăng từ 0,095 lên 0,156 →
  MAE (trên điểm) và macro F1 (trên nhãn) **không phải lúc nào cũng cùng chiều**, nên cần xem cả hai.
- Từ k = 7 trở đi, cả hai độ đo **bão hoà**: **Top-9** đạt **93%** mức cải thiện MAE (0,1801 so với 0,1787 — chỉ chênh
  0,0014) và macro F1 0,347 so với 0,352 (**98,5%**), accuracy 0,601 so với 0,607, với **60%** số đặc trưng.
- Thêm `day` sau `month` (k = 5) không thay đổi gì — hai đặc trưng dư thừa của nhau (mục 4.2). 6 đặc trưng cuối
  ($R < 0{,}05$) chỉ đóng góp ~7% mức cải thiện.
- Linear Regression train rất nhanh (vài chục ms) nên chênh lệch thời gian không đáng kể; khác biệt về tốc độ rõ hơn
  với Logistic Regression (mục 5.5).

### 5.2 Chọn theo ngưỡng: giữ các đặc trưng có $R \ge t$

In [ ]:
thr_df = pd.DataFrame([evaluate(f"R ≥ {t}", [f for f in RANKING if score.loc[f, "R"] >= t])[0]
                       for t in (0.2, 0.1, 0.09, 0.05, 0.04, 0.0)])
thr_df["% cải thiện"] = improvement(thr_df["MAE"])
thr_df.set_index("tập đặc trưng")[["số đặc trưng", "MAE", "accuracy", "macro F1", "% cải thiện", "train_time_s", "đặc trưng"]]

**Nhận xét (5.2):** Ngưỡng càng cao → càng ít đặc trưng → MAE càng tăng. Ngưỡng $R \ge 0{,}1$ giữ 7 đặc trưng và đạt
87% mức cải thiện; ngưỡng $R \ge 0{,}05$ giữ 10 đặc trưng, đạt 93% (bằng Top-9 — đặc trưng thứ 10 `looped` không thêm
gì). Ngưỡng 0,2 chỉ giữ 2 đặc trưng → quá ít (macro F1 0,156).

### 5.3 Loại đặc trưng dư thừa (tương quan mạnh với nhau)

In [ ]:
red_rows = []
for thr in (None, 0.95, 0.9, 0.8, 0.7):
    kept = drop_redundant(FEATURES, thr) if thr else FEATURES
    row = evaluate(f"Loại dư thừa |r_ij| ≥ {thr}" if thr else "Không loại", kept)[0]
    row["đặc trưng bị loại"] = ", ".join(f for f in RANKING if f not in kept) or "—"
    red_rows.append(row)
for thr in (0.95, 0.9):                     # kết hợp: Top-K_BEST rồi loại dư thừa
    kept = drop_redundant(RANKING[:K_BEST], thr)
    row = evaluate(f"Top-{K_BEST} + loại dư thừa |r_ij| ≥ {thr}", kept)[0]
    row["đặc trưng bị loại"] = ", ".join(f for f in RANKING[:K_BEST] if f not in kept) or "—"
    red_rows.append(row)
red_df = pd.DataFrame(red_rows)
red_df["% cải thiện"] = improvement(red_df["MAE"])
red_df.set_index("tập đặc trưng")[["số đặc trưng", "MAE", "accuracy", "macro F1", "% cải thiện", "đặc trưng bị loại"]]

**Nhận xét (5.3):**
- Bỏ `day` (trùng `month`, r = 0,996) → MAE, accuracy, macro F1 **không đổi** → `day` hoàn toàn dư thừa. Kết hợp
  Top-9 + bỏ `day` được **8 đặc trưng** với kết quả y hệt Top-9.
- Bỏ thêm `income` (ngưỡng 0,9, vì r(`income`, `income_per_neighbor`) = 0,95) làm macro F1 **giảm rõ** (0,352 → 0,317):
  dù tương quan rất mạnh, hai đặc trưng vẫn mang thông tin riêng (khác nhau ở các địa chỉ có nhiều giao dịch đến) —
  bảng hệ số ở mục 5.6 cho thấy mô hình dùng **hiệu** của chúng (hệ số trái dấu, cùng độ lớn).
- → Ngưỡng dư thừa nên đặt **cao** (≈ 0,95); ngưỡng thấp hơn loại nhầm đặc trưng có ích.

### 5.4 Đối chứng

Nếu tương quan thực sự đo được "độ quan trọng", thì cùng số lượng đặc trưng nhưng chọn các đặc trưng có $R$
**thấp nhất** phải cho kết quả kém hơn hẳn. Để hai tập **không trùng nhau**, ta so sánh Top-$m$ với $m$ đặc trưng
$R$ thấp nhất, trong đó $m$ = số đặc trưng bị Top-`K_BEST` loại bỏ (tức các đặc trưng thấp nhất chính là những
đặc trưng ta đề xuất bỏ đi). So sánh thêm Top-`K_BEST` theo $\eta$ (= `SelectKBest(f_classif)`).

In [ ]:
ctrl = pd.DataFrame([evaluate(f"Top-{K_BEST} theo R", RANKING[:K_BEST])[0],
                     evaluate(f"Top-{K_BEST} theo η (SelectKBest f_classif)", ETA_RANKING[:K_BEST])[0],
                     evaluate(f"Top-{N_CTRL} theo R", RANKING[:N_CTRL])[0],
                     evaluate(f"{N_CTRL} đặc trưng R thấp nhất (= các đặc trưng Top-{K_BEST} bỏ đi)", RANKING[-N_CTRL:])[0],
                     evaluate("Top-2 theo R", RANKING[:2])[0],
                     evaluate("2 đặc trưng R thấp nhất", RANKING[-2:])[0], BASE])
ctrl["% cải thiện"] = improvement(ctrl["MAE"])
ctrl.set_index("tập đặc trưng")[["số đặc trưng", "MAE", "accuracy", "macro F1", "% cải thiện", "đặc trưng"]]

**Nhận xét (5.4):**
- 6 đặc trưng có $R$ **thấp nhất** (chính là 6 đặc trưng Top-9 bỏ đi) chỉ đạt 21% mức cải thiện MAE, accuracy 0,500
  và macro F1 0,095 — **bằng baseline**: mô hình vẫn luôn đoán `white`. Trong khi Top-6 đạt 77% và macro F1 0,305.
- 2 đặc trưng $R$ thấp nhất (`length`, `count`): 1,8% — gần như không có thông tin; Top-2: 69%.
- Top-9 theo $\eta$ (`SelectKBest(f_classif)`) là **cùng một tập** với Top-9 theo $R$ → hai thước đo tương quan
  thống nhất.
- → Tương quan **xác định đúng** đặc trưng quan trọng và đặc trưng nên bỏ.

### 5.5 Kiểm chứng chéo với một bộ phân lớp tuyến tính khác: Logistic Regression

Linear Regression (bình phương tối thiểu) là bộ phân lớp tuyến tính đơn giản nhất. **Logistic Regression** (softmax)
cũng là bộ phân lớp tuyến tính nhưng tối ưu log-loss và cho xác suất thật sự ($\in [0, 1]$, tổng bằng 1). Nếu tập đặc
trưng được chọn bằng tương quan cũng tốt với Logistic Regression thì kết quả chọn đặc trưng **không phụ thuộc** vào
một mô hình cụ thể. MAE được tính giữa xác suất `predict_proba` và nhãn one-hot.

In [ ]:
def evaluate_logreg(name, cols):
    cols = list(cols)
    model = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])
    t0 = time.perf_counter(); model.fit(X_train[cols], y_train); t_train = time.perf_counter() - t0
    P = model.predict_proba(X_test[cols]); pred = P.argmax(axis=1)
    return {"tập đặc trưng": name, "số đặc trưng": len(cols), "MAE": mean_absolute_error(Y_test, P),
            "accuracy": accuracy_score(y_test, pred), "macro F1": f1_score(y_test, pred, average="macro"),
            "train_time_s": t_train}

no_dup = drop_redundant(FEATURES, 0.95)
BEST_SET = drop_redundant(RANKING[:K_BEST], 0.95)      # tập đề xuất: Top-K_BEST rồi bỏ đặc trưng trùng thông tin
subsets = {f"Tất cả {len(FEATURES)} đặc trưng": FEATURES,
           f"Loại dư thừa ≥ 0.95": no_dup,
           f"Top-{K_BEST} theo R": RANKING[:K_BEST],
           f"Top-{K_BEST} + loại dư thừa ≥ 0.95": BEST_SET,
           "Top-2 theo R": RANKING[:2],
           f"{N_CTRL} đặc trưng R thấp nhất": RANKING[-N_CTRL:]}
cross = pd.concat({"LinearRegression": pd.DataFrame([evaluate(n, c)[0] for n, c in subsets.items()]).set_index("tập đặc trưng"),
                   "LogisticRegression": pd.DataFrame([evaluate_logreg(n, c) for n, c in subsets.items()]).set_index("tập đặc trưng")},
                  axis=1)
cross.loc[:, (slice(None), ["số đặc trưng", "MAE", "accuracy", "macro F1", "train_time_s"])]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
for ax, metric in zip(axes, ["MAE", "macro F1"]):
    cross.xs(metric, axis=1, level=1).plot.barh(ax=ax, color=["tab:blue", "tab:purple"])
    ax.set(title=f"{metric} trên tập test — hai bộ phân lớp tuyến tính", xlabel=metric, ylabel="")
    ax.invert_yaxis()
axes[1].get_legend().remove()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/linreg_vs_logreg.png", dpi=120); plt.show()

**Nhận xét (5.5):**
- Với Logistic Regression, thứ tự các tập đặc trưng **giữ nguyên**: tập đề xuất 8 đặc trưng đạt macro F1 0,385 so với
  0,396 khi dùng cả 15 (97%), còn 6 đặc trưng $R$ thấp nhất chỉ 0,105.
- Thời gian train Logistic Regression với 8 đặc trưng **nhanh khoảng 2–3 lần** so với 15 đặc trưng (số giây cụ thể
  dao động giữa các lần chạy).
- Logistic Regression tốt hơn Linear Regression một chút (macro F1 0,396 so với 0,352; MAE 0,144 so với 0,179) vì nó
  tối ưu trực tiếp xác suất lớp, nhưng cả hai đều là bộ phân lớp tuyến tính nên cùng giới hạn ở mức ~0,6 accuracy.

### 5.6 Tổng hợp

In [ ]:
rows = [BASE, ALL,
        evaluate("Loại dư thừa |r_ij| ≥ 0.95", no_dup)[0],
        evaluate(f"Top-{K_BEST} theo R", RANKING[:K_BEST])[0],
        evaluate(f"Top-{K_BEST} + loại dư thừa ≥ 0.95", BEST_SET)[0],
        evaluate("Top-2 theo R", RANKING[:2])[0],
        evaluate("Top-1 theo R", RANKING[:1])[0],
        evaluate(f"Đối chứng: {N_CTRL} đặc trưng R thấp nhất", RANKING[-N_CTRL:])[0]]
summary = pd.DataFrame(rows)
summary["% cải thiện"] = improvement(summary["MAE"])
summary["MAE chênh so với tất cả"] = summary["MAE"] - ALL["MAE"]

colors = ["tab:gray" if n == 0 else "tab:red" if name.startswith("Đối chứng") else
          "tab:green" if name.startswith(f"Top-{K_BEST}") else "tab:blue"
          for name, n in zip(summary["tập đặc trưng"], summary["số đặc trưng"])]
labels = [f"{a}  [{n} đặc trưng]" for a, n in zip(summary["tập đặc trưng"], summary["số đặc trưng"])]
fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
for ax, metric, fmt in zip(axes, ["MAE", "macro F1"], ["{:.4f}", "{:.3f}"]):
    vals = summary[metric][::-1]
    ax.barh(labels[::-1], vals, color=colors[::-1])
    for i, v in enumerate(vals):
        ax.text(v, i, " " + fmt.format(v), va="center", fontsize=9)
    ax.margins(x=0.15)
axes[0].set(xlim=(summary["MAE"].min() - 0.01, summary["MAE"].max() + 0.008),
            xlabel="MAE trên tập test (càng nhỏ càng tốt)", title="MAE của Linear Regression (phân lớp)")
axes[1].set(xlabel="macro F1 trên tập test (càng lớn càng tốt)", title="macro F1")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/mae_subsets.png", dpi=120); plt.show()
summary.set_index("tập đặc trưng")[["số đặc trưng", "MAE", "MAE chênh so với tất cả", "accuracy", "macro F1",
                                    "% cải thiện", "train_time_s", "đặc trưng"]]

In [ ]:
_, model_best = evaluate("best", BEST_SET)
coef = pd.DataFrame(model_best.named_steps["lr"].coef_.T, index=BEST_SET, columns=CLASSES)
pred_best = model_best.predict(X_test[BEST_SET]).argmax(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(19, 6.5), gridspec_kw={"width_ratios": [1, 1.1]})
sns.heatmap(coef, annot=True, fmt="+.2f", cmap="RdBu_r", center=0, annot_kws={"size": 8},
            cbar_kws={"label": "hệ số w (đặc trưng đã chuẩn hoá)"}, ax=axes[0])
axes[0].set(title=f"Hệ số Linear Regression — mỗi cột là một đầu ra ŝ_k (tập đề xuất, {len(BEST_SET)} đặc trưng)", xlabel="", ylabel="")
axes[0].tick_params(axis="x", rotation=30)
ConfusionMatrixDisplay.from_predictions(y_test, pred_best, display_labels=CLASSES, normalize="true",
                                        values_format=".2f", cmap="Blues", ax=axes[1], colorbar=False)
axes[1].set_title(f"Ma trận nhầm lẫn (chuẩn hoá theo hàng = recall) — tập đề xuất")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/coef_confusion.png", dpi=120); plt.show()

In [ ]:
# Xuất kết quả cho demo Streamlit (proj1/data/bai2_cls)
APP_DIR = "proj1/data/bai2_cls"
os.makedirs(APP_DIR, exist_ok=True)
EN = {"tập đặc trưng": "subset", "đặc trưng thêm vào": "added", "số đặc trưng": "n_features", "đặc trưng": "features",
      "% cải thiện": "improvement_pct", "MAE chênh so với tất cả": "mae_vs_all", "macro F1": "macro_f1"}
(corr_cls.loc[RANKING].assign(R=score["R"], eta=eta, rank=score["hạng"])
 .rename_axis("feature").reset_index().to_csv(f"{APP_DIR}/corr_classes.csv", index=False))
corr_ff.loc[RANKING, RANKING].to_csv(f"{APP_DIR}/corr_matrix.csv")
topk.rename(columns=EN).to_csv(f"{APP_DIR}/topk.csv", index=False)
summary.rename(columns=EN).to_csv(f"{APP_DIR}/subsets.csv", index=False)
(cross.stack(level=0, future_stack=True).rename_axis(["subset", "model"]).reset_index()
 .rename(columns=EN).to_csv(f"{APP_DIR}/cross.csv", index=False))
# Toàn bộ tập cân bằng (đúng phép chia train/test ở trên) → tab "Thử nghiệm trực tiếp" tái lập đúng kết quả notebook
name = dict(enumerate(CLASSES))
pd.concat([X_train.assign(target=y_train.map(name), split="train"),
           X_test.assign(target=y_test.map(name), split="test")]).to_csv(
    f"{APP_DIR}/sample.csv.gz", index=False, float_format="%.6g")
print(sorted(os.listdir(APP_DIR)))

**Nhận xét (5.6):**
- Hệ số lớn nhất thuộc về cặp `income` (+) và `income_per_neighbor` (−) với dấu ngược nhau ở mỗi lớp → mô hình dùng
  **hiệu** $\log(\text{income}) - \log(\text{income/neighbors}) \approx \log(\text{neighbors})$ (khi neighbors > 1).
- Ma trận nhầm lẫn: `white` được nhận đúng 89%, `princetonCerber` 61%, `princetonLocky` 54%; nhưng
  `montrealCryptXXX` bị nhầm sang `princetonCerber` (cùng năm 2016) và `otherRansom` (gộp nhiều họ khác nhau) không
  được nhận ra → ranh giới **tuyến tính** không đủ tách các lớp chồng lấn; Bài 1 cần mô hình cây (XGBoost) cho việc
  này.

## 6. Kết luận

| Tập đặc trưng | Số đặc trưng | MAE (test) | % cải thiện MAE | Accuracy | Macro F1 |
|---|---|---|---|---|---|
| Baseline (không dùng đặc trưng) | 0 | 0,1986 | 0% | 0,500 | 0,095 |
| Tất cả | 15 | 0,1787 | 100% | 0,607 | 0,352 |
| Loại dư thừa (bỏ `day`) | 14 | 0,1786 | 100% | 0,607 | 0,351 |
| Top-9 theo $R$ (= `SelectKBest(f_classif, 9)`) | 9 | 0,1801 | 92,9% | 0,601 | 0,347 |
| **Top-9 + bỏ `day` (đề xuất)** | **8** | **0,1801** | **92,9%** | **0,601** | **0,346** |
| Top-2 (`income_tz`, `year`) | 2 | 0,1849 | 68,8% | 0,521 | 0,156 |
| Đối chứng: 6 đặc trưng $R$ thấp nhất | 6 | 0,1943 | 21,3% | 0,500 | 0,095 |

1. **Bài toán & mô hình:** giữ đúng bài toán **phân lớp 7 lớp** của Bài 1; Linear Regression được dùng như bộ phân
   lớp tuyến tính (one-hot + argmax, tương đương `RidgeClassifier`), đánh giá bằng MAE giữa điểm dự đoán và nhãn
   one-hot, kèm accuracy và macro F1.
2. **Phương pháp tương quan cho phân lớp:** tương quan **point-biserial** giữa đặc trưng và biến chỉ thị từng lớp,
   tổng hợp thành $R_j$ (relevance) + loại cặp đặc trưng tương quan mạnh với nhau (redundancy). Xếp hạng gần như trùng
   với tỉ số tương quan $\eta$ / `SelectKBest(f_classif)` của scikit-learn.
3. **Tập đề xuất — 8 đặc trưng:** `income_tz, year, income_per_neighbor, month, income, neighbors, weight,
   looped_ratio`. Dùng **53%** số đặc trưng mà MAE chỉ tăng 0,0014, macro F1 đạt 98% so với dùng cả 15; Logistic
   Regression train nhanh khoảng 2–3 lần với cùng tập này.
4. **Đặc trưng dư thừa:** `day` (trùng `month`) bỏ đi không ảnh hưởng. Ngưỡng dư thừa phải cao (0,95): bỏ `income` vì
   tương quan 0,95 với `income_per_neighbor` lại làm giảm macro F1.
5. **Kiểm chứng:** 6 đặc trưng $R$ thấp nhất cho kết quả bằng baseline; kết luận giữ nguyên khi đổi sang Logistic
   Regression → tương quan phản ánh đúng mức độ quan trọng của đặc trưng.
6. **Hạn chế:** cần tập thử nghiệm **cân bằng** (trên phân phối thật, bộ phân lớp bình phương tối thiểu luôn đoán
   `white`); bộ phân lớp tuyến tính chỉ đạt accuracy ~0,6 / macro F1 ~0,35 (XGBoost ở Bài 1 tốt hơn nhiều). Tương quan
   chỉ xét **từng cặp** và chỉ đo quan hệ **tuyến tính** (ví dụ `year` có quan hệ không đơn điệu với các họ
   ransomware) → có thể bổ sung Mutual Information (`mutual_info_classif`), RFE hoặc `SelectFromModel`.